# 🎬 IMDB Movie Review Sentiment Analysis
### Multi-Model Comparison Pipeline
**Dataset:** IMDB 50K Reviews | **Task:** Binary Sentiment Classification (Positive / Negative)

| Model | Vectoriser |
|-------|-----------|
| Logistic Regression | TF-IDF bigrams |
| Linear SVM | TF-IDF bigrams |
| Naive Bayes | TF-IDF bigrams |
| Random Forest | TF-IDF unigrams |
| Gradient Boosting | Bag-of-Words |

## ⚙️ Step 1 — Install & Import Libraries

In [1]:
# All core packages come pre-installed on Colab; wordcloud needs installing
!pip install wordcloud -q

import pandas as pd
import numpy as np
import re, time, warnings, os
warnings.filterwarnings("ignore")

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)

# Viz
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 📂 Step 2 — Load Dataset
**Option A** — Upload directly in Colab  
**Option B** — Mount Google Drive (recommended if the file is already in Drive)

In [ ]:
# ── OPTION A: Upload from your computer ──────────────────────────────────
from google.colab import files

print("Upload your 'IMDB Dataset.csv' file:")
uploaded = files.upload()
csv_path = list(uploaded.keys())[0]
print(f"\nUsing file: {csv_path}")

Upload your 'IMDB Dataset.csv' file:


TypeError: 'NoneType' object is not subscriptable

In [7]:
# ── Load the CSV ─────────────────────────────────────────────────────────
df = pd.read_csv(csv_path)
df.columns = [c.strip().lower() for c in df.columns]

print(f"Shape       : {df.shape}")
print(f"Columns     : {df.columns.tolist()}")
print(f"\nSentiment counts:")
print(df['sentiment'].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum())
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IMDB Dataset.csv'

## 🔍 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("IMDB Dataset — EDA", fontsize=15, fontweight='bold')

# 1. Sentiment distribution
counts = df['sentiment'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#2EC4B6', '#E71D36'], startangle=90,
            textprops={'fontsize': 12})
axes[0].set_title('Sentiment Distribution')

# 2. Review word count
df['word_count'] = df['review'].apply(lambda x: len(str(x).split()))
for sentiment, color in [('positive', '#2EC4B6'), ('negative', '#E71D36')]:
    subset = df[df['sentiment'] == sentiment]['word_count']
    axes[1].hist(subset.clip(upper=700), bins=60, alpha=0.65,
                 color=color, label=sentiment, edgecolor='none')
axes[1].set_xlabel('Word Count (clipped at 700)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Review Length Distribution')
axes[1].legend()

# 3. Top 20 word counts bar
from collections import Counter
import re
all_words = re.sub(r'[^a-z ]', ' ',
                   ' '.join(df['review'].str.lower().tolist())).split()
stop = {'the','a','and','of','to','is','in','it','this','that','was',
        'for','on','are','with','as','an','be','have','i','film','movie',
        'br','but','not','my','his','her','they','were','he','she','at',
        'from','by','or','its','had','one','all','when','who','so','there',
        'has','very','which','more','just','like','up','do','me','would',
        'their','been','out','if','what','about','time','can','some','see'}
top20 = Counter(w for w in all_words if w not in stop and len(w) > 2).most_common(20)
words, freqs = zip(*top20)
axes[2].barh(list(reversed(words)), list(reversed(freqs)),
             color='#4361EE', edgecolor='none')
axes[2].set_title('Top 20 Most Frequent Words')
axes[2].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('eda.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 EDA complete!")

In [ ]:
# ── Word Clouds ───────────────────────────────────────────────────────────
from wordcloud import WordCloud

def make_cloud(text, title, colormap, ax):
    wc = WordCloud(width=600, height=300, background_color='white',
                   colormap=colormap, max_words=150,
                   stopwords=stop).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
pos_text = ' '.join(df[df['sentiment'] == 'positive']['review'].tolist())
neg_text = ' '.join(df[df['sentiment'] == 'negative']['review'].tolist())
make_cloud(pos_text, '☀️  Positive Reviews', 'YlGn',   ax1)
make_cloud(neg_text, '🌧️  Negative Reviews', 'OrRd', ax2)
plt.suptitle('Word Clouds by Sentiment', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## 🧹 Step 4 — Text Preprocessing

In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r'<[^>]+>', ' ', text)       # strip HTML tags (e.g. <br />)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)  # keep letters only
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)           # collapse whitespace
    return text

df['clean'] = df['review'].apply(clean_text)
df['label'] = (df['sentiment'] == 'positive').astype(int)

print("Before cleaning:")
print(df['review'].iloc[0][:200])
print("\nAfter cleaning:")
print(df['clean'].iloc[0][:200])
print(f"\nLabel encoding: positive=1, negative=0")
print(df['label'].value_counts())

## ✂️ Step 5 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['label'],
    test_size=0.20,
    random_state=42,
    stratify=df['label']       # keeps 50/50 balance in both splits
)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"\nClass balance in test set:")
print(pd.Series(y_test).value_counts())

## 🤖 Step 6 — Define Model Pipelines
Each pipeline is a **Vectoriser → Classifier** pair that sklearn handles as one unit.

In [ ]:
PIPELINES = {
    "Logistic Regression\n(TF-IDF)": Pipeline([
        ("vec", TfidfVectorizer(max_features=60_000, ngram_range=(1, 2),
                                sublinear_tf=True, min_df=3)),
        ("clf", LogisticRegression(C=5, max_iter=1000, solver='lbfgs')),
    ]),
    "Linear SVM\n(TF-IDF)": Pipeline([
        ("vec", TfidfVectorizer(max_features=60_000, ngram_range=(1, 2),
                                sublinear_tf=True, min_df=3)),
        ("clf", LinearSVC(C=1.0, max_iter=2000)),
    ]),
    "Naive Bayes\n(TF-IDF)": Pipeline([
        ("vec", TfidfVectorizer(max_features=40_000, ngram_range=(1, 2),
                                sublinear_tf=False, min_df=3)),
        ("clf", MultinomialNB(alpha=0.1)),
    ]),
    "Random Forest\n(TF-IDF)": Pipeline([
        ("vec", TfidfVectorizer(max_features=20_000, ngram_range=(1, 1),
                                sublinear_tf=True, min_df=5)),
        ("clf", RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                       random_state=42)),
    ]),
    "Gradient Boosting\n(Bag-of-Words)": Pipeline([
        ("vec", CountVectorizer(max_features=15_000, ngram_range=(1, 1),
                                min_df=5)),
        ("clf", GradientBoostingClassifier(n_estimators=150,
                                           learning_rate=0.1,
                                           max_depth=4,
                                           random_state=42)),
    ]),
}

print(f"✅ {len(PIPELINES)} model pipelines defined and ready to train.")

## 🏋️ Step 7 — Train All Models & Evaluate

In [ ]:
results  = {}
trained  = {}
PALETTE  = ['#2EC4B6', '#E71D36', '#FF9F1C', '#4361EE', '#7B2FBE']

for name, pipe in PIPELINES.items():
    short = name.replace('\n', ' ')
    print(f"▶ Training: {short} ...", end=" ", flush=True)
    t0 = time.time()
    pipe.fit(X_train, y_train)
    elapsed = time.time() - t0

    y_pred = pipe.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)

    try:
        y_score = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_score = pipe.decision_function(X_test)

    auc    = roc_auc_score(y_test, y_score)
    report = classification_report(y_test, y_pred,
                                   target_names=['negative', 'positive'],
                                   output_dict=True)
    cm     = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_score)

    results[name] = dict(accuracy=acc, auc=auc, train_time=elapsed,
                         report=report, cm=cm, fpr=fpr, tpr=tpr, y_score=y_score)
    trained[name] = pipe
    print(f"✓  Accuracy={acc:.4f}  AUC={auc:.4f}  ({elapsed:.1f}s)")

print("\n✅ All models trained!")

## 📊 Step 8 — Results Summary Table

In [ ]:
summary = pd.DataFrame([
    {
        'Model': n.replace('\n', ' '),
        'Accuracy': f"{r['accuracy']:.4f}",
        'ROC-AUC':  f"{r['auc']:.4f}",
        'Train Time (s)': f"{r['train_time']:.1f}",
        'F1 (positive)': f"{r['report']['positive']['f1-score']:.4f}",
        'F1 (negative)': f"{r['report']['negative']['f1-score']:.4f}",
    }
    for n, r in sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True)
])

# Style the table
summary.style.background_gradient(subset=['Accuracy', 'ROC-AUC'], cmap='YlGn')

## 📈 Step 9 — Visualisations

In [ ]:
names   = list(results.keys())
labels  = [n.replace('\n', ' ') for n in names]
accs    = [results[n]['accuracy']   for n in names]
aucs    = [results[n]['auc']        for n in names]
times   = [results[n]['train_time'] for n in names]

plt.style.use('seaborn-v0_8-whitegrid')
fig = plt.figure(figsize=(20, 14))
fig.suptitle('IMDB Sentiment Analysis — Model Comparison', fontsize=18,
             fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.4)

# (a) Accuracy
ax1 = fig.add_subplot(gs[0, :2])
bars = ax1.barh(labels, accs, color=PALETTE, edgecolor='none', height=0.5)
for bar, v in zip(bars, accs):
    ax1.text(v - 0.001, bar.get_y() + bar.get_height()/2,
             f'{v:.4f}', va='center', ha='right', fontsize=9,
             fontweight='bold', color='white')
ax1.set_xlim(min(accs) - 0.04, 1.01)
ax1.set_title('Test Accuracy', fontweight='bold')
ax1.set_xlabel('Accuracy')

# (b) AUC
ax2 = fig.add_subplot(gs[0, 2:])
bars2 = ax2.barh(labels, aucs, color=PALETTE, edgecolor='none', height=0.5)
for bar, v in zip(bars2, aucs):
    ax2.text(v - 0.001, bar.get_y() + bar.get_height()/2,
             f'{v:.4f}', va='center', ha='right', fontsize=9,
             fontweight='bold', color='white')
ax2.set_xlim(min(aucs) - 0.04, 1.01)
ax2.set_title('ROC-AUC Score', fontweight='bold')
ax2.set_xlabel('AUC')

# (c) Training time
ax3 = fig.add_subplot(gs[1, :2])
ax3.bar(range(len(labels)), times, color=PALETTE, edgecolor='none', width=0.5)
ax3.set_xticks(range(len(labels)))
ax3.set_xticklabels(labels, rotation=15, ha='right', fontsize=8)
ax3.set_ylabel('Seconds')
ax3.set_title('Training Time', fontweight='bold')

# (d) ROC curves
ax4 = fig.add_subplot(gs[1, 2:])
for i, name in enumerate(names):
    ax4.plot(results[name]['fpr'], results[name]['tpr'],
             color=PALETTE[i], lw=2,
             label=f"{labels[i]} (AUC={results[name]['auc']:.3f})")
ax4.plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
ax4.set_xlabel('False Positive Rate')
ax4.set_ylabel('True Positive Rate')
ax4.set_title('ROC Curves', fontweight='bold')
ax4.legend(fontsize=7)

# (e) Top 2 confusion matrices
top2 = sorted(names, key=lambda n: results[n]['accuracy'], reverse=True)[:2]
for j, name in enumerate(top2):
    ax = fig.add_subplot(gs[2, j*2:(j+1)*2])
    sns.heatmap(results[name]['cm'], annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative','Positive'],
                yticklabels=['Negative','Positive'],
                ax=ax, cbar=False, annot_kws={'size':13,'weight':'bold'})
    ax.set_title(f"Confusion Matrix\n{name.replace(chr(10),' ')}",
                 fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard saved as model_comparison.png")

In [ ]:
# ── Per-model F1 / Precision / Recall heatmap ────────────────────────────
fig2, axes = plt.subplots(1, len(names), figsize=(20, 4))
fig2.suptitle('Precision / Recall / F1 per Model', fontsize=14,
              fontweight='bold')
for i, (name, ax) in enumerate(zip(names, axes)):
    rpt  = results[name]['report']
    vals = np.array([[rpt[cls][m] for m in ['precision','recall','f1-score']]
                     for cls in ['negative','positive']])
    sns.heatmap(vals, annot=True, fmt='.3f', cmap='YlGn',
                vmin=0.80, vmax=1.0,
                xticklabels=['Prec','Recall','F1'],
                yticklabels=['Neg','Pos'],
                ax=ax, cbar=False,
                annot_kws={'size':11,'weight':'bold'})
    ax.set_title(f"{labels[i]}\nAcc={results[name]['accuracy']:.4f}",
                 fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig('classification_reports.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔬 Step 10 — Deep Dive: Top Features (Logistic Regression)

In [ ]:
# Extract the top positive / negative words learned by Logistic Regression
lr_pipe = trained['Logistic Regression\n(TF-IDF)']
vec     = lr_pipe.named_steps['vec']
clf     = lr_pipe.named_steps['clf']

feature_names = np.array(vec.get_feature_names_out())
coefs         = clf.coef_[0]

top_n = 20
top_pos_idx  = np.argsort(coefs)[-top_n:][::-1]
top_neg_idx  = np.argsort(coefs)[:top_n]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Logistic Regression — Top Predictive Features',
             fontsize=14, fontweight='bold')

# Positive features
ax1.barh(feature_names[top_pos_idx][::-1],
         coefs[top_pos_idx][::-1],
         color='#2EC4B6', edgecolor='none')
ax1.set_title('Top 20 → Positive Sentiment', fontweight='bold')
ax1.set_xlabel('Coefficient value')

# Negative features
ax2.barh(feature_names[top_neg_idx][::-1],
         np.abs(coefs[top_neg_idx][::-1]),
         color='#E71D36', edgecolor='none')
ax2.set_title('Top 20 → Negative Sentiment', fontweight='bold')
ax2.set_xlabel('|Coefficient value|')

plt.tight_layout()
plt.savefig('top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎯 Step 11 — Predict on Your Own Reviews

In [ ]:
# Select the best model automatically
best_key = max(results, key=lambda n: results[n]['accuracy'])
best_pipe = trained[best_key]
print(f"🏆 Using: {best_key.replace(chr(10), ' ')}  "
      f"(accuracy={results[best_key]['accuracy']:.4f})")

def predict_sentiment(reviews, pipe=best_pipe):
    cleaned = [clean_text(r) for r in reviews]
    preds   = pipe.predict(cleaned)
    try:
        probs = pipe.predict_proba(cleaned)[:, 1]
    except AttributeError:
        scores = pipe.decision_function(cleaned)
        probs  = 1 / (1 + np.exp(-scores))   # sigmoid for LinearSVC

    print("\n" + "="*70)
    for rev, pred, prob in zip(reviews, preds, probs):
        label = "✅ POSITIVE" if pred == 1 else "❌ NEGATIVE"
        conf  = prob if pred == 1 else 1 - prob
        print(f"{label}  (confidence: {conf:.1%})")
        print(f"  Review: {rev[:100]}...")
        print()

# ✏️  Add your own reviews here!
my_reviews = [
    "This film was an absolute masterpiece! The acting was phenomenal and the story kept me on the edge of my seat.",
    "Terrible movie. The plot made no sense, the acting was wooden, and it was a complete waste of two hours.",
    "It was okay. Some parts were interesting but overall a bit disappointing given the hype surrounding it.",
    "One of the greatest films I have ever seen. Every scene was beautifully crafted and emotionally moving.",
    "I walked out halfway through. Boring, predictable, and poorly executed in every single way possible.",
]

predict_sentiment(my_reviews)

## 💾 Step 12 — Save Best Model

In [ ]:
import pickle

model_filename = 'imdb_sentiment_model.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(best_pipe, f)
print(f"✅ Model saved as '{model_filename}'")

# Download it to your computer
from google.colab import files
files.download(model_filename)

In [ ]:
# ── Load & use the saved model later ─────────────────────────────────────
# with open('imdb_sentiment_model.pkl', 'rb') as f:
#     loaded_model = pickle.load(f)
#
# review = "This movie was absolutely fantastic!"
# pred   = loaded_model.predict([clean_text(review)])[0]
# print("Positive" if pred == 1 else "Negative")